# Satellite Land-Use Classification & Image Reconstruction

UOC Deep Learning (M2.875, 2025–2026). CNNs, EfficientNetB0, transfer learning, fine-tuning, autoencoder and PyTorch.

Source: augmented UC Merced dataset (https://www.kaggle.com/datasets/apollo2506/landuse-scene-classification). Download the images separately. This adaptation preserves the experimental code but removes academic prompts, paths and execution outputs. Results in ../docs/MODEL_RESULTS.md were not independently reproduced.

## Environment and imports

In [ ]:
# Importamos tensorflow
import tensorflow as tf
print("TF version   : ", tf.__version__)

# Necesitaremos GPU
print("GPU available: ", tf.config.list_physical_devices('GPU'))

# keras version is 2.11.0
import keras
print("Keras version   : ", keras.__version__)

In [ ]:
# Importamos los elementos de keras que utilizaremos con mayor frecuencia
from keras.utils import image_dataset_from_directory
from keras.layers import (
    GlobalAveragePooling2D, Flatten,
    Dense, Dropout, Conv2D, Conv2DTranspose, BatchNormalization,
    MaxPooling2D, UpSampling2D, Rescaling, Resizing)
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam
from keras import Sequential, Model

In [ ]:
# Importamos el resto de librerías que necesitaremos para la PEC
import cv2
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
from PIL import Image
from tqdm.auto import tqdm
import json

from tensorflow.keras.layers import Rescaling

# Procesamiento paralelo
from concurrent.futures import ThreadPoolExecutor


## Dataset location

In [ ]:
# Portable dataset configuration. Set LANDUSE_DATA_DIR if needed.
from pathlib import Path
import os
configured_dir = os.environ.get("LANDUSE_DATA_DIR")
if configured_dir:
    dataset_root = Path(configured_dir).expanduser().resolve()
else:
    rel = Path("data/external/landuse-scene-classification")
    candidates = [Path.cwd() / rel, Path.cwd().parent / rel]
    dataset_root = next((p.resolve() for p in candidates if p.is_dir()), candidates[0].resolve())
if not dataset_root.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_root}. Download the Kaggle dataset and/or set LANDUSE_DATA_DIR.")
data_dir = str(dataset_root)
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
print("Dataset root:", data_dir)


## Dataset exploration and labels

In [ ]:
# Extraer nombres de las clases



# AJUSTAR ESTA RUTA SEGÚN CORRESPONDA PARA ACCEDER A LOS DIRECTORIOS
# data_dir is configured in the preceding cell

# Se eligió train.csv pero se puede cambiar a cualquier otro poniendo el nombre}
# como parámetro de os.path.join(data_dir, "nombreArchivo.csv")

csv_file = os.path.join(data_dir, "train.csv")
json_file = os.path.join(data_dir, "label_map.json")

df = pd.read_csv(csv_file)
print("Columnas:", list(df.columns))

# Heurística: posibles nombres de columna de clase
candidates = ["label", "class", "category", "target", "class_name", "classname", "name",
              "label_id", "class_id", "category_id", "target_id", "Label", "ClassName"]

label_col = None
for c in candidates:
    if c in df.columns:
        label_col = c
        break
# Control de errores
if label_col is None:
    raise ValueError("No se encontró una columna de clase. Revisar las columnas impresas arriba.")

# Caso A: ya viene como texto (nombres)
if df[label_col].dtype == "object":
    classes = sorted(df[label_col].dropna().unique())
    print(f"\n Clases encontradas en '{label_col}' (total {len(classes)}):")
    for i, cls in enumerate(classes, 1):
        print(f"{i:02d}. {cls}")

# Caso B: viene como id numérico, usamos label_map.json
else:
    if not os.path.exists(json_file):
        raise ValueError(f"'{label_col}' parece ser numérico, pero no existe {json_file} para mapear ids a nombres.")
    with open(json_file, "r", encoding="utf-8") as f:
        label_map = json.load(f)

    # Se normaliza keys por si vienen como strings
    # label_map puede ser { "0": "cat", ... } o { "cat": 0, ... }
    # Se detecta orientación:
    sample_key = next(iter(label_map.keys()))
    if str(sample_key).isdigit():  # {id: name}
        id_to_name = {int(k): v for k, v in label_map.items()}
    else:  # {name: id}
        id_to_name = {int(v): k for k, v in label_map.items()}

    ids = sorted(df[label_col].dropna().unique())
    classes = [id_to_name[i] for i in ids]

    print(f"\n Clases encontradas vía '{label_col}' + label_map.json (total {len(classes)}):")
    for i, cls in enumerate(classes, 1):
        print(f"{i:02d}. {cls}")


In [ ]:
# Número de instancias por conjunto

# Para cada conjunto (train / validation / test), el número de instancias es
# simplemente el número de filas del CSV (sin contar el header).


files = {
    "train": "train.csv",
    "validation": "validation.csv",
    "test": "test.csv",
}

for split, fname in files.items():
    path = os.path.join(data_dir, fname)
    df = pd.read_csv(path)
    print(f"{split:10s}: {len(df):,} instancias  |  columnas: {df.shape[1]}")

In [ ]:
# Número de instancias por clase


files = {
    "train": "train.csv",
    "validation": "validation.csv",
    "test": "test.csv",
}

# Usar nombres de clase
label_col = "ClassName"

for split, fname in files.items():
    df = pd.read_csv(os.path.join(data_dir, fname))
    counts = df[label_col].value_counts().sort_index()

    print(f"\n=== {split.upper()} ===")
    print(f"Total instancias: {len(df)}")
    print(counts)

    mn, mx = counts.min(), counts.max()
    print(f"min={mn} | max={mx} | max/min={mx/mn:.3f}")

## Image-directory verification

In [ ]:
# Esta es una celda común para los tres puntos solicitados.



CSV_PATHS = {
    "train": os.path.join(data_dir, "train.csv"),
    "validation": os.path.join(data_dir, "validation.csv"),
    "test": os.path.join(data_dir, "test.csv"),
}

# Se detectará la carpeta de imágenes

IMG_ROOT_CANDIDATES = [
    os.path.join(data_dir, "images_train_test_val"),
    os.path.join(data_dir, "images"),
    data_dir,
]

def resolve_split_root(split_name: str, example_relpath: str) -> str:
    """
    Devuelve la ruta base para el split (train/validation/test) tal que:
    os.path.join(split_root, example_relpath) exista.

    NOTA: Función reusable para los 3 conjuntos solicitados.
    """
    for root in IMG_ROOT_CANDIDATES:
        # Caso A: root ya contiene las imágenes del split directamente
        p = os.path.join(root, example_relpath)
        if os.path.exists(p):
            return root

        # Caso B: root/split contiene las imágenes
        p2 = os.path.join(root, split_name, example_relpath)
        if os.path.exists(p2):
            return os.path.join(root, split_name)

        # Caso C: validation a veces es "val"
        if split_name == "validation":
            p3 = os.path.join(root, "val", example_relpath)
            if os.path.exists(p3):
                return os.path.join(root, "val")

    raise FileNotFoundError(
        "No se puede detectar la carpeta de imágenes automáticamente.\n"
    )

def count_images_in_folder_by_class(split_root: str, classes: list, ext=(".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp")):
    """
    Cuenta imágenes por clase leyendo carpetas:
    split_root/<ClassName>/*.(ext)

    NOTA: Función reusable para los 3 conjuntos solicitados.
    """
    counts = {}
    for c in classes:
        class_dir = os.path.join(split_root, c)
        n = 0
        if os.path.isdir(class_dir):
            for e in ext:
                n += len(glob.glob(os.path.join(class_dir, f"*{e}")))
        counts[c] = n
    return pd.Series(counts).sort_index()


def dynamic_range_from_filelist(file_list):
    """
    Calcula min y max de píxeles recorriendo imágenes.

    OPTIMIZACIÓN: paraleliza lectura/decodificación con threads y procesa en chunks grandes.

    NOTA: Función reusable para los 3 conjuntos solicitados.
    """
    gmin, gmax = None, None

    # Ajustes (chunk grande + threads)
    WORKERS = 64   # threads para I/O + decodificación
    print("Número de hilos:" + str(WORKERS))
    ## Se aumentó el chunk de 512 a 1024 para mejorar el tiempo de procesamiento
    CHUNK_SIZE = 1024 # chunk grande

    def minmax_one(fp):
        # Abrir y cerrar dentro del thread
        with Image.open(fp) as im:
            arr = np.asarray(im)  # Convertir a array
        return int(arr.min()), int(arr.max())

    with tqdm(total=len(file_list), desc="Calculando rango dinámico") as pbar:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            for i in range(0, len(file_list), CHUNK_SIZE):
                batch = file_list[i:i+CHUNK_SIZE]
                for mn, mx in ex.map(minmax_one, batch):
                    gmin = mn if gmin is None else min(gmin, mn)
                    gmax = mx if gmax is None else max(gmax, mx)
                pbar.update(len(batch))

    return gmin, gmax

def show_one_image_per_class(split_root: str, classes: list, ncols=7):
    """
    Visualiza 1 imagen por clase (la primera encontrada en la carpeta).

    NOTA: Función reusable para los 3 conjuntos solicitados.
    """
    n = len(classes)
    nrows = (n + ncols - 1) // ncols

    plt.figure(figsize=(3*ncols, 3*nrows))
    for i, c in enumerate(classes):
        class_dir = os.path.join(split_root, c)
        img_files = []
        for e in (".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"):
            img_files += glob.glob(os.path.join(class_dir, f"*{e}"))

        ax = plt.subplot(nrows, ncols, i+1)
        if img_files:
            img = Image.open(img_files[0])
            ax.imshow(img)
        ax.set_title(c, fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

def analyze_split(split_name: str):
    """
    - Cuenta por clase usando CSV (esperado)
    - Cuenta por clase usando carpetas (real)
    - Comprueba coincidencia
    - Visualiza 1 imagen por clase
    - Rango dinámico min/max

    NOTA: Función reusable para los 3 conjuntos solicitados.
    """
    df = pd.read_csv(CSV_PATHS[split_name])

    # Columnas reales
    CLASS_COL = "ClassName"
    FILE_COL  = "Filename"

    classes = sorted(df[CLASS_COL].unique())

    # Detectar split_root a partir de una ruta real en el CSV
    example_rel = df[FILE_COL].iloc[0]  # ej: runway/runway_000259.png
    split_root = resolve_split_root(split_name, example_rel)

    # Conteo por clase según CSV
    csv_counts = df[CLASS_COL].value_counts().sort_index()

    # Conteo por clase según carpetas
    folder_counts = count_images_in_folder_by_class(split_root, classes)

    # Comparación de conteo de clases según CSV <-> clase según carpetas
    comp = pd.DataFrame({
        "csv": csv_counts.reindex(classes).fillna(0).astype(int),
        "carpeta": folder_counts.reindex(classes).fillna(0).astype(int),
    })
    comp["diff(carpeta-csv)"] = comp["carpeta"] - comp["csv"]

    print(f"\n=== {split_name.upper()} ===")
    print("Split root:", split_root)
    print("Total CSV:", len(df), "| Total carpeta:", int(comp["carpeta"].sum()))
    display(comp)

    mism = comp[comp["diff(carpeta-csv)"] != 0]
    if len(mism) == 0:
        print(" Coincide EXACTO carpeta vs CSV para todas las clases.")
    else:
        print("Hay diferencias. Revisar estas clases:")
        display(mism)

    # Visualizar una imagen por clase
    print("\n Ejemplo: 1 imagen por clase")
    show_one_image_per_class(split_root, classes)

    # Rango dinámico: se calculará leyendo SOLO las imágenes listadas en el CSV
    file_list = [os.path.join(split_root, rel) for rel in df[FILE_COL].tolist()]
    # Asegurar de que existan (por si hay rutas raras)
    file_list = [fp for fp in file_list if os.path.exists(fp)]

    mn, mx = dynamic_range_from_filelist(file_list)
    print(f"\n Rango dinámico ({split_name}): min={mn}, max={mx}")

In [ ]:

# Para el conjunto de datos de entrenamiento
analyze_split("train")

In [ ]:
# Para el conjunto de datos de validación
analyze_split("validation")

In [ ]:
# Para el conjunto de datos de test
analyze_split("test")

## TensorFlow input pipeline

In [ ]:
# RUTAS
BASE_IMG_DIR = os.path.join(data_dir, "images_train_test_val")
TRAIN_DIR = os.path.join(BASE_IMG_DIR, "train")
VAL_DIR   = os.path.join(BASE_IMG_DIR, "validation")
TEST_DIR  = os.path.join(BASE_IMG_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

def make_ds(dir_path, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        dir_path,
        labels="inferred",
        label_mode="int",
        color_mode="rgb",
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,          # Se redimensiona a 224x224
        shuffle=shuffle,
        seed=SEED
    )
    # Mantener rango dinámico: image_dataset_from_directory devuelve float32 en [0,255]
    ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_data = make_ds(TRAIN_DIR, shuffle=True)
val_data   = make_ds(VAL_DIR, shuffle=False)
test_data  = make_ds(TEST_DIR, shuffle=False)

# MODIFICACIÓN AÑADIDA (aceleración I/O y pipeline)
# Si se desea pasar a la ejecución de las épocas SIN OPTIMIZACIÓN elimine las
# siguientes tres (3) líneas de código

train_data = train_data.cache().prefetch(tf.data.AUTOTUNE)
val_data   = val_data.cache().prefetch(tf.data.AUTOTUNE)
test_data  = test_data.cache().prefetch(tf.data.AUTOTUNE)

#############################################################################

print("Datasets creados y optimizados con cache+prefetch:")
print("train:", train_data)
print("val:  ", val_data)
print("test: ", test_data)

# Verificar rango dinámico real en un batch
for x, y in train_data.take(1):
    print("Batch shape:", x.shape, y.shape)
    print("dtype:", x.dtype)
    print("min:", tf.reduce_min(x).numpy(), "max:", tf.reduce_max(x).numpy())

## Dense baseline

In [ ]:
# Definición de la red



num_classes = len(classes)

dense_model = tf.keras.Sequential([
    tf.keras.Input(shape=(224, 224, 3)),
    tf.keras.layers.Resizing(32, 32),
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1024, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

dense_model.summary()

In [ ]:
# Compilación de la red

opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

dense_model.compile(
    optimizer=opt,
    loss="sparse_categorical_crossentropy",   # etiquetas int (0..20)
    metrics=["accuracy"]
)

In [ ]:
# Entrenamiento de la red

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_dense = dense_model.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Plot del training loss y la accuracy



def plot_history(history, title_prefix=""):
    # Accuracy
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.title(f"{title_prefix}Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["train", "val"])
    plt.show()

    # Loss
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.title(f"{title_prefix}Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["train", "val"])
    plt.show()

plot_history(history_dense, title_prefix="Dense NN - ")


best_epoch = int(np.argmin(history_dense.history["val_loss"])) + 1
best_val = min(history_dense.history["val_loss"])
print("Mejor época:", best_epoch, "mejor val_loss:", best_val)
print("Épocas entrenadas:", len(history_dense.history["loss"]))

## Custom convolutional neural network

In [ ]:
# Definición de la red


num_classes = len(classes)

inputs = tf.keras.Input(shape=(224, 224, 3), name="input_224x224x3")
x = tf.keras.layers.Rescaling(1./255, name="rescale_0_1")(inputs)

# Conv(5x5, 16) + MaxPool
x = tf.keras.layers.Conv2D(16, (5,5), padding="same", activation="relu", name="conv1_5x5_16")(x)
x = tf.keras.layers.MaxPooling2D(name="pool1")(x)

# Conv(3x3, 32) + MaxPool
x = tf.keras.layers.Conv2D(32, (3,3), padding="same", activation="relu", name="conv2_3x3_32")(x)
x = tf.keras.layers.MaxPooling2D(name="pool2")(x)

# Conv(3x3, 64) + MaxPool
x = tf.keras.layers.Conv2D(64, (3,3), padding="same", activation="relu", name="conv3_3x3_64")(x)
x = tf.keras.layers.MaxPooling2D(name="pool3")(x)

# GAP -> vector 64
x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)

# Clasificador
x = tf.keras.layers.Dense(1024, activation="relu", name="dense_1024")(x)
x = tf.keras.layers.Dropout(0.5, name="dropout_0_5")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="output_softmax")(x)

cnn_small = tf.keras.Model(inputs=inputs, outputs=outputs, name="CNN_small")

cnn_small.summary()

In [ ]:
# Compilación de la red


opt = tf.keras.optimizers.Adam(learning_rate=0.001)

cnn_small.compile(
    optimizer=opt,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Entrenamiento


early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

start_time = time.time()

history_cnn_small = cnn_small.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)

train_time_sec = time.time() - start_time
print(f"Tiempo total entrenamiento: {train_time_sec/60:.2f} min")
print(f"Épocas entrenadas: {len(history_cnn_small.history['loss'])}")

In [ ]:
# Resultados

def plot_history(history, title_prefix=""):
    # Accuracy
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.title(f"{title_prefix}Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["train", "val"])
    plt.show()

    # Loss
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.title(f"{title_prefix}Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["train", "val"])
    plt.show()

plot_history(history_cnn_small, title_prefix="CNN pequeña - ")

test_loss, test_acc = cnn_small.evaluate(test_data, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

best_epoch = int(np.argmin(history_cnn_small.history["val_loss"])) + 1
best_val_loss = float(np.min(history_cnn_small.history["val_loss"]))
best_val_acc = float(np.max(history_cnn_small.history["val_accuracy"]))

print(f"Mejor época (por val_loss): {best_epoch}")
print(f"Mejor val_loss: {best_val_loss:.4f}")
print(f"Mejor val_accuracy: {best_val_acc:.4f}")

### Original test result

Custom CNN: **86.57% accuracy**.

## Convolutional autoencoder

In [ ]:
# data rescalling
normalization_layer = Rescaling(1./255)

normalized_train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
normalized_val_data = val_data.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
train_data_auto = normalized_train_data.map(lambda x, y: (x, x))
val_data_auto = normalized_val_data.map(lambda x, y: (x, x))

In [ ]:
image_batch, label_batch = iter(train_data_auto).get_next()
print("Las dimensiones de un batch de imágenes es: {}".format(image_batch.shape))
print("Las dimensiones de un batch de etiquetas es: {}".format(label_batch.shape))

In [ ]:
first_image = image_batch[0]
print("En la primera imagen los valores mínimo y máximo son {} y {}, respectivamente"
      .format(np.min(first_image),np.max(first_image)))

In [ ]:
# Definición de la red


# 1) Reescalado externo (0,1)
normalization_layer = Rescaling(1./255)

normalized_train_data = train_data.map(lambda x, y: (normalization_layer(x), y))
normalized_val_data   = val_data.map(lambda x, y: (normalization_layer(x), y))

# 2) Autoencoder: objetivo = la propia imagen
train_data_auto = normalized_train_data.map(lambda x, y: (x, x))
val_data_auto   = normalized_val_data.map(lambda x, y: (x, x))

# 3) Comprobaciones pedidas
image_batch, label_batch = iter(train_data_auto).get_next()
print("Las dimensiones de un batch de imágenes es: {}".format(image_batch.shape))
print("Las dimensiones de un batch de etiquetas es: {}".format(label_batch.shape))

first_image = image_batch[0].numpy()
print("En la primera imagen los valores mínimo y máximo son {} y {}, respectivamente"
      .format(np.min(first_image), np.max(first_image)))

# 4) Definición del autoencoder (Functional API)
inputs = tf.keras.Input(shape=(224, 224, 3), name="ae_input")

# ----- ENCODER -----
x = tf.keras.layers.Conv2D(16, (5,5), padding="same", activation="relu", name="enc_conv1_5x5_16")(inputs)
x = tf.keras.layers.MaxPooling2D(name="enc_pool1")(x)

x = tf.keras.layers.Conv2D(32, (3,3), padding="same", activation="relu", name="enc_conv2_3x3_32")(x)
x = tf.keras.layers.MaxPooling2D(name="enc_pool2")(x)

x = tf.keras.layers.Conv2D(64, (3,3), padding="same", activation="relu", name="enc_conv3_3x3_64")(x)
x = tf.keras.layers.MaxPooling2D(name="enc_pool3")(x)

# ----- DECODER -----
x = tf.keras.layers.Conv2DTranspose(64, (3,3), padding="same", activation="relu", name="dec_deconv1_3x3_64")(x)
x = tf.keras.layers.UpSampling2D(name="dec_upsample1")(x)

x = tf.keras.layers.Conv2DTranspose(32, (3,3), padding="same", activation="relu", name="dec_deconv2_3x3_32")(x)
x = tf.keras.layers.UpSampling2D(name="dec_upsample2")(x)

x = tf.keras.layers.Conv2DTranspose(16, (5,5), padding="same", activation="relu", name="dec_deconv3_5x5_16")(x)
x = tf.keras.layers.UpSampling2D(name="dec_upsample3")(x)

# Capa final: 3 filtros + sigmoide (salida en [0,1])
outputs = tf.keras.layers.Conv2D(3, (3,3), padding="same", activation="sigmoid", name="ae_output")(x)

autoencoder = tf.keras.Model(inputs, outputs, name="autoencoder_cnn")

autoencoder.summary()

In [ ]:
# Compilación de la red

opt = tf.keras.optimizers.Adam(learning_rate=0.001)

autoencoder.compile(
    optimizer=opt,
    loss="mse"   # error cuadrático medio
)

In [ ]:
# Entrenamiento

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history_ae = autoencoder.fit(
    train_data_auto,
    validation_data=val_data_auto,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)

print(f"Épocas entrenadas: {len(history_ae.history['loss'])}")

In [ ]:
# Representación del loss


import matplotlib.pyplot as plt
import numpy as np

plt.figure()
plt.plot(history_ae.history["loss"])
plt.plot(history_ae.history["val_loss"])
plt.title("Autoencoder - Loss (MSE)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["train", "val"])
plt.show()

best_epoch = int(np.argmin(history_ae.history["val_loss"])) + 1
best_val_loss = float(np.min(history_ae.history["val_loss"]))
print(f"Mejor época (por val_loss): {best_epoch}")
print(f"Mejor val_loss: {best_val_loss:.6f}")

In [ ]:
# Normalización de los datos


# Reescalado del conjunto de test a (0,1)
normalized_test_data = test_data.map(lambda x, y: (normalization_layer(x), y))

# Dataset autoencoder: objetivo = la propia imagen
test_data_auto = normalized_test_data.map(lambda x, y: (x, x))

# Comprobación rápida (opcional)
image_batch, label_batch = iter(test_data_auto).get_next()
print("Batch imágenes:", image_batch.shape)
print("Batch etiquetas:", label_batch.shape)



In [ ]:
# Evaluación del modelo


test_mse = autoencoder.evaluate(test_data_auto, verbose=1)
print(f"Test MSE (loss): {test_mse:.6f}")

### Original reconstruction result

Convolutional autoencoder: **0.001367 test MSE**.

In [ ]:
# Visualización de los datos

# Tomar un batch del test
x_batch, _ = next(iter(test_data_auto))   # x_batch está en [0,1]

# Reconstrucción
x_recon = autoencoder.predict(x_batch, verbose=0)  # también en [0,1]

# Para visualizar: convertir a enteros [0,255]
x_orig_vis = np.clip(x_batch.numpy() * 255.0, 0, 255).astype(np.uint8)
x_recon_vis = np.clip(x_recon * 255.0, 0, 255).astype(np.uint8)

# Mostrar 4 parejas (8 imágenes en total)
n = 4
plt.figure(figsize=(10, 2.5*n))

for i in range(n):
    # Original
    ax = plt.subplot(n, 2, 2*i + 1)
    ax.imshow(x_orig_vis[i])
    ax.set_title(f"Original {i+1}")
    ax.axis("off")

    # Reconstruida
    ax = plt.subplot(n, 2, 2*i + 2)
    ax.imshow(x_recon_vis[i])
    ax.set_title(f"Reconstruida {i+1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## EfficientNetB0 transfer learning

In [ ]:
from tensorflow.keras.applications import EfficientNetB0

# Cargar el modelo base preentrenado


num_classes = len(classes)

base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False  # congelar pesos

In [ ]:
# Definición de la red

inputs = tf.keras.Input(shape=(224, 224, 3), name="input_224x224x3")

x = base_model(inputs, training=False)  # importante: training=False al estar congelado
x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
x = tf.keras.layers.Dense(50, activation="relu", name="dense_50")(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="output_softmax")(x)

effnet_model = tf.keras.Model(inputs, outputs, name="EfficientNetB0_TL")

effnet_model.summary()

In [ ]:
# Compilación de la red

opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

effnet_model.compile(
    optimizer=opt,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Entrenamiento de la red

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=10,
    mode="max",
    restore_best_weights=True
)

start_time = time.time()

history_eff = effnet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=[early_stop],
    verbose=1
)

train_time_sec = time.time() - start_time
print(f"Tiempo total entrenamiento: {train_time_sec/60:.2f} min")
print(f"Épocas entrenadas: {len(history_eff.history['loss'])}")

In [ ]:
# Resultados


def plot_history(history, title_prefix=""):
    # Accuracy
    plt.figure()
    plt.plot(history.history["accuracy"])
    plt.plot(history.history["val_accuracy"])
    plt.title(f"{title_prefix}Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(["train", "val"])
    plt.show()

    # Loss
    plt.figure()
    plt.plot(history.history["loss"])
    plt.plot(history.history["val_loss"])
    plt.title(f"{title_prefix}Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(["train", "val"])
    plt.show()

plot_history(history_eff, title_prefix="EfficientNetB0 TL - ")

test_loss, test_acc = effnet_model.evaluate(test_data, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

best_epoch_acc = int(np.argmax(history_eff.history["val_accuracy"])) + 1
best_val_acc = float(np.max(history_eff.history["val_accuracy"]))
best_val_loss = float(np.min(history_eff.history["val_loss"]))

print(f"Mejor época (por val_accuracy): {best_epoch_acc}")
print(f"Mejor val_accuracy: {best_val_acc:.4f}")
print(f"Mejor val_loss: {best_val_loss:.4f}")

### Original test result

Frozen-backbone EfficientNetB0: **97.90% accuracy**.

## EfficientNetB0 fine-tuning

In [ ]:
# Definición del modelo

# Fine-tuning: descongelar el modelo base
base_model.trainable = True

print("base_model.trainable =", base_model.trainable)
effnet_model.summary()

In [ ]:
# Compilación de la red

opt_ft = tf.keras.optimizers.Adam(learning_rate=1e-5)

effnet_model.compile(
    optimizer=opt_ft,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Entrenamiento

start_time = time.time()

history_eff_ft = effnet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    verbose=1
)

train_time_sec = time.time() - start_time
print(f"Tiempo fine-tuning: {train_time_sec/60:.2f} min")

In [ ]:
# Resultados

plot_history(history_eff_ft, title_prefix="EfficientNetB0 Fine-tuning - ")

test_loss_ft, test_acc_ft = effnet_model.evaluate(test_data, verbose=1)
print(f"Test loss (fine-tune): {test_loss_ft:.4f}")
print(f"Test accuracy (fine-tune): {test_acc_ft:.4f}")

best_epoch = int(np.argmax(history_eff_ft.history["val_accuracy"])) + 1
best_val_acc = float(np.max(history_eff_ft.history["val_accuracy"]))
print(f"Mejor época (val_accuracy): {best_epoch}")
print(f"Mejor val_accuracy (10 ep): {best_val_acc:.4f}")

### Original test result

EfficientNetB0 after fine-tuning: **98.10% accuracy**.

## PyTorch CNN

In [ ]:
# Imports y paths

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_dir = os.path.join(data_dir, "images_train_test_val", "train")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Carga de datos (ImageFolder + DataLoader)

BATCH_SIZE = 32

train_tfms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),  # -> float32 en [0,1]
])

train_ds = datasets.ImageFolder(root=train_dir, transform=train_tfms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

print("Clases detectadas:", len(train_ds.classes))
print("Ejemplo clases:", train_ds.classes[:5])

In [ ]:
# Definición del modelo (nn.Module)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=21):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),  # 224x224 -> 224x224
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 112x112 -> 112x112
            nn.ReLU(),
            nn.MaxPool2d(2),                              # -> 56x56
        )
        self.gap = nn.AdaptiveAvgPool2d((1, 1))           # -> (32, 1, 1)
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)   # (B, 32)
        x = self.classifier(x)    # (B, 21)
        return x

model = SimpleCNN(num_classes=21).to(device)
print(model)

In [ ]:
# Loss y optimizador

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Bucle de entrenamiento (5 épocas)

EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch}/{EPOCHS} - loss: {epoch_loss:.4f}")

## Limitations

Original metrics were not independently reproduced. Augmentations might share parent scenes across train/test. No image dataset or model weights are published. The PyTorch example does not provide a test accuracy.